In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('job_us_sample.csv')
df.head()

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate,shift,site_name,skills,uniq_id
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago,Telecommuting not available|Travel not required,NaN,SEE BELOW,418ff92580b270ef4e7c14f0ddfc36b4
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago,Telecommuting not available|Travel not required,NaN,"linux/unix, network monitoring, incident respo...",8aec88cba08d53da65ab99cf20f6f9d9
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago,Telecommuting not available|Travel not required,NaN,"Enterprise Solutions Architecture, business in...",46baa1f69ac07779274bcd90b85d9a72
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago,Telecommuting not available|Travel not required,NaN,Please see job description,3941b2f206ae0f900c4fba4ac0b18719
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago,Telecommuting not available|Travel not required,NaN,"Configuration Management, Developer, Linux, Ma...",45efa1f6bc65acc32bbbb953a1ed13b7


In [53]:
print(df.columns.tolist())

['advertiserurl', 'company', 'employmenttype_jobstatus', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'postdate', 'shift', 'site_name', 'skills', 'uniq_id']


In [56]:
df.shape

(22000, 12)

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   advertiserurl             22000 non-null  object
 1   company                   21950 non-null  object
 2   employmenttype_jobstatus  21770 non-null  object
 3   jobdescription            22000 non-null  object
 4   jobid                     22000 non-null  object
 5   joblocation_address       21997 non-null  object
 6   jobtitle                  22000 non-null  object
 7   postdate                  22000 non-null  object
 8   shift                     21643 non-null  object
 9   site_name                 3490 non-null   object
 10  skills                    21957 non-null  object
 11  uniq_id                   22000 non-null  object
dtypes: object(12)
memory usage: 2.0+ MB


In [58]:
#Selected Important Features
df = df[['jobtitle', 'skills', 'jobdescription', 'advertiserurl']]
df.head()

,jobtitle,skills,jobdescription,advertiserurl
0,AUTOMATION TEST ENGINEER,SEE BELOW,Looking for Selenium engineers...must have sol...,https://www.dice.com/jobs/detail/AUTOMATION-TE...
1,Information Security Engineer,"linux/unix, network monitoring, incident respo...",The University of Chicago has a rapidly growin...,https://www.dice.com/jobs/detail/Information-S...
2,Business Solutions Architect,"Enterprise Solutions Architecture, business in...","GalaxE.SolutionsEvery day, our solutions affec...",https://www.dice.com/jobs/detail/Business-Solu...
3,"Java Developer (mid level)- FT- GREAT culture,...",Please see job description,Java DeveloperFull-time/direct-hireBolingbrook...,https://www.dice.com/jobs/detail/Java-Develope...
4,DevOps Engineer,"Configuration Management, Developer, Linux, Ma...",Midtown based high tech firm has an immediate ...,https://www.dice.com/jobs/detail/DevOps-Engine...


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   jobtitle        22000 non-null  object
 1   skills          21957 non-null  object
 2   jobdescription  22000 non-null  object
 3   advertiserurl   22000 non-null  object
dtypes: object(4)
memory usage: 687.6+ KB


In [60]:
#Handle Missing Values
df['jobtitle'] = df['jobtitle'].fillna('')
df['skills'] = df['skills'].fillna('')
df['jobdescription'] = df['jobdescription'].fillna('')
df['advertiserurl'] = df['advertiserurl'].fillna('')

In [61]:
df.isna().sum()

jobtitle          0
skills            0
jobdescription    0
advertiserurl     0
dtype: int64

In [62]:
#Combining the given features 
df['combine_text'] = (df['jobtitle']+' '+df['skills']+' '+df['jobdescription']+' '+df['advertiserurl'])

In [64]:
df['combine_text']

0        AUTOMATION TEST ENGINEER SEE BELOW Looking for...
1        Information Security Engineer linux/unix, netw...
2        Business Solutions Architect Enterprise Soluti...
3        Java Developer (mid level)- FT- GREAT culture,...
4        DevOps Engineer Configuration Management, Deve...
                               ...                        
21995    Web Designer UI/UX mobile apps, interaction de...
21996    Senior Front End Web Developer - Full Time at ...
21997    QA Analyst SDLC, ALM, SQL, T-SQL, RedGate, Tea...
21998    Tech Lead-Full Stack Python, Ruby, Go, Clojure...
21999    C/C++ Programmer Null Experience in C/C++ Prog...
Name: combine_text, Length: 22000, dtype: object

In [65]:
# Convert Text to Numbers with TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combine_text'])
print(tfidf_matrix.shape)

(22000, 165768)


#Calculate Similarity
similarity = cosine_similarity(tfidf_matrix)

similarity

In [67]:
#Recommendation function
def recommend_jobs(user_skills):
    #convert input skills into vector 
    input_vector = tfidf.transform([user_skills])

    #Calculate similarity with all jobs
    similarity_scores = cosine_similarity(input_vector, tfidf_matrix)

    #Convert to 1D array
    scores = similarity_scores.flatten()

    top_indices = scores.argsort()[-5:][::-1]

    #Display recommendations
    recommendations = df.iloc[top_indices][['jobtitle', 'skills', 'advertiserurl']]

    return recommendations
    

In [71]:
#test the model 
recommendations = recommend_jobs('Python Machine Learning SQL Data Analusis')

for count, (_,row) in enumerate(
    recommendations.iterrows(), start=1):
    
    print(f'\nRecommendation {count}')
    print('Job Title :', row['jobtitle'])
    print('Skills    :', row['skills'])
    print('Apply URL :', row['advertiserurl'])

    print('======'*10)


Recommendation 1
Job Title : Data Scientist - NYC
Skills    : Analysis, Automated, Data Mining, Data Modeling, Development, Java, Matlab, Modeling, Perl, Python, Research, Validation
Apply URL : https://www.dice.com/jobs/detail/Data-Scientist-%2526%252345-NYC-Amazon-New-York-NY-10001/amazon20/418984?icid=sr9445-315p&q=&l=New%20York,%20NY

Recommendation 2
Job Title : Data Scientist - Houston
Skills    : Analysis, Automated, Data Mining, Data Modeling, Development, Java, Matlab, Modeling, Perl, Python, Research, Validation
Apply URL : https://www.dice.com/jobs/detail/Data-Scientist-%2526%252345-Houston-Amazon-Houston-TX-77001/amazon20/418983?icid=sr33-2p&q=&l=Houston,%20TX

Recommendation 3
Job Title : Machine Learning Specialist
Skills    : Machine Learning, ML, ICML, NIPS, KDD, CVPR, ACL, C#, Java, Python, Apache, Hadoop, Apache Spark, MPI, CUDA
Apply URL : https://www.dice.com/jobs/detail/Machine-Learning-Specialist-Zensa-Redmond-WA-98052/90954501/827955?icid=sr65802-2194p&q=&l=Cali